# Cross-cell Evaluation Notebook

用于评估某个细胞系训练得到的模型在本细胞系和其他细胞系 test split 上的表现。对应正式脚本：

```text
evaluate_cross_cell.py
```


## Step 0. 参数设置


In [3]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "lamar_m6a").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

/picb/rnasys2/lidaoqian/LAMAR-M6A


In [10]:
MODE = "full"  # full / mature
MODEL_CELL = "a549"
TARGET_CELLS = ["a549", "cd8t", "hek293", "hela", "molm13"]
LENGTH = 801
SPLIT = "test"

DATA_DIR = Path("data")
OUTPUTS_DIR = Path("outputs")
CHECKPOINT_TEMPLATE = "outputs/finetune/{mode}/{model_cell}-{length}-bs16-acc4"
CHECKPOINT_PATH = Path("outputs/finetune/full/a549-101-bs16-acc4/checkpoint-3100")  # 可填具体 checkpoint 文件或目录，例如 Path("outputs/checkpoints/full/a549-801-bs16-acc4/checkpoint-3100")

TOKENIZER_PATH = Path("tokenizer/single_nucleotide")
MODEL_CONFIG = Path("configs/config_150M.json")
MODEL_CLASS = "scripts.sequence_classification_patch:EsmForSequenceClassification"
MODEL_MAX_LENGTH = LENGTH + 2
INFER_BATCH_SIZE = 128
DEVICE = None  # None 表示自动选择 cuda:0 / cpu


## Step 1. 导入函数


In [11]:
from argparse import Namespace
from lamar_m6a.evaluation.evaluate_cross_cell import (
    build_checkpoint_path,
    compute_metrics,
    csv_path_for,
    find_model_state,
    load_model,
    predict_csv,
)
from transformers import AutoTokenizer
import pandas as pd


## Step 2. 检查输入 CSV 和 checkpoint


In [12]:
args = Namespace(
    mode=MODE,
    model_cell=MODEL_CELL,
    model_cells=None,
    target_cells=TARGET_CELLS,
    length=LENGTH,
    split=SPLIT,
    data_dir=DATA_DIR,
    input_csv=None,
    outputs_dir=OUTPUTS_DIR,
    checkpoint_template=CHECKPOINT_TEMPLATE,
    checkpoint_path=CHECKPOINT_PATH,
    tokenizer_path=TOKENIZER_PATH,
    model_config=MODEL_CONFIG,
    model_class=MODEL_CLASS,
    model_max_length=MODEL_MAX_LENGTH,
    padding_side="left",
    use_fast_tokenizer=False,
    num_labels=2,
    token_dropout=False,
    positional_embedding_type="rotary",
    hidden_size=768,
    intermediate_size=3072,
    num_attention_heads=12,
    num_hidden_layers=12,
    head_type="Linear",
    kernel_sizes=None,
    ocs=None,
    strict_load=True,
    batch_size=16,
    gradient_accumulation_steps=4,
    infer_batch_size=INFER_BATCH_SIZE,
    device=DEVICE,
)

checkpoint_root = CHECKPOINT_PATH or build_checkpoint_path(args, MODEL_CELL, LENGTH)
model_state_path = find_model_state(checkpoint_root)
print("checkpoint:", model_state_path)

for cell in TARGET_CELLS:
    path = csv_path_for(DATA_DIR, MODE, SPLIT, cell, LENGTH)
    print(cell, path, path.exists())


checkpoint: outputs/finetune/full/a549-101-bs16-acc4/checkpoint-3100/model.safetensors
a549 data/processed/full_splits/test_m6a_full_a549_801.csv True
cd8t data/processed/full_splits/test_m6a_full_cd8t_801.csv True
hek293 data/processed/full_splits/test_m6a_full_hek293_801.csv True
hela data/processed/full_splits/test_m6a_full_hela_801.csv True
molm13 data/processed/full_splits/test_m6a_full_molm13_801.csv False


## Step 3. 加载模型和 tokenizer


In [13]:
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH,
    model_max_length=MODEL_MAX_LENGTH,
    padding_side="left",
    use_fast=False,
)
model = load_model(args, model_state_path, tokenizer)
model


EsmForSequenceClassification(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(10, 768, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(1026, 768, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-11): 12 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((768,), eps=1e-0

## Step 4. 逐个目标细胞系预测并汇总 metrics


In [14]:
summary_rows = []
for target_cell in TARGET_CELLS:
    csv_path = csv_path_for(DATA_DIR, MODE, SPLIT, target_cell, LENGTH)
    if not csv_path.exists():
        print("skip missing:", csv_path)
        continue
    output_path = OUTPUTS_DIR / "predictions" / "cross_cell" / MODE / f"model-{MODEL_CELL}_target-{target_cell}_{LENGTH}_{SPLIT}_pred.csv"
    metrics = predict_csv(args, model, tokenizer, csv_path, output_path)
    print(target_cell, metrics)
    if metrics is not None:
        row = {"mode": MODE, "model_cell": MODEL_CELL, "target_cell": target_cell, "length": LENGTH, "split": SPLIT, "prediction_csv": str(output_path)}
        row.update(metrics)
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUTS_DIR / "metrics" / "cross_cell" / MODE / f"model-{MODEL_CELL}_{LENGTH}_{SPLIT}.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
summary_df


test_m6a_full_a549_801.csv:   0%|                                                     | 1/971 [00:05<1:32:40,  5.73s/it]


KeyboardInterrupt: 